# EDA — Fase 3: exploración y descomposición\n\n**chronolab** · `notebooks/01_eda.ipynb`\n\nEsta notebook cubre la Fase 3 del plan del proyecto: perfil de calidad de datos, descomposición estacional, autocorrelación, perfiles agregados, relación con temperatura y estadísticos de dificultad de la serie. El resumen ejecutable de lo que aquí se encuentra vive en [`docs/EDA_FINDINGS.md`](../docs/EDA_FINDINGS.md); las fases siguientes leen ese archivo, no esta notebook.\n\n## ⚠️ Aviso sobre los datos: esta ejecución usa la fuente sintética de demostración\n\nEsta notebook debe poder ejecutarse **de principio a fin sin red**, usando la caché local. Como todavía no existe una descarga real de UCI/REE/Open-Meteo cacheada en `data/raw/`, esta ejecución usa `chronolab.data.sources.synthetic.SyntheticElectricitySource` y `SyntheticWeatherSource`: tres series de demanda con una dificultad de predicción deliberadamente escalonada (`residential_north` fácil, `commercial_mixed` media, `volatile_industrial` difícil) más una temperatura horaria consistente con ellas, generadas en hora local de Madrid con las mismas trampas de cambio de hora que tendría una fuente real, y con huecos, duplicados, ceros y atípicos inyectados a propósito para que el perfil de calidad tenga algo genuino que reportar.\n\n**Todo lo que aparece aquí describe el pipeline y el método — no son hallazgos reales sobre demanda eléctrica.** El código que ejecuta cada sección es exactamente el que se ejecutará contra UCI/REE cuando haya red disponible: basta con sustituir `SyntheticElectricitySource`/`SyntheticWeatherSource` por `UCIElectricitySource`/`OpenMeteoSource` (o `REEDemandSource`) en la celda de carga. `docs/EDA_FINDINGS.md` repite este aviso en la cabecera.\n\n## Qué cubre cada sección\n\n1. Perfil de calidad: cobertura, huecos, duplicados, ceros, atípicos, continuidad tras el cambio de hora.\n2. Descomposición MSTL (24, 168): tendencia, ambas estacionalidades, residuo.\n3. ACF/PACF hasta el retardo 200, más periodograma.\n4. Perfiles agregados: hora × día de la semana, efecto de festivos, perfil mensual.\n5. Relación demanda-temperatura: forma en U, grados-día de calefacción/refrigeración.\n6. Estadísticos de dificultad: fuerza de tendencia, fuerza estacional, entropía espectral.\n\nLas funciones reutilizables están en [`chronolab.viz.plots`](../src/chronolab/viz/plots.py) (figuras) y [`chronolab.data.quality`](../src/chronolab/data/quality.py) (perfil de calidad); esta notebook las orquesta y narra, no reimplementa nada."

In [1]:
from pathlib import Path

import pandas as pd
import plotly.io as pio

from chronolab.data.align import deduplicate, reindex_to_full_grid, to_utc_naive
from chronolab.data.cache import CachedSource
from chronolab.data.calendar import calendar_features
from chronolab.data.quality import coverage_report, detect_outliers, dst_transition_report
from chronolab.data.sources.synthetic import (
    DEMO_SERIES_IDS,
    SyntheticElectricitySource,
    SyntheticWeatherSource,
)
from chronolab.viz import plots

# Raiz del repo: esta notebook vive en notebooks/, un nivel por debajo.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FIGURES_DIR = REPO_ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.precision", 3)
pio.templates.default = "plotly_white"

# Rango completo de la ventana de demostracion sintetica (14 meses; ver el
# docstring de chronolab.data.sources.synthetic para por que ese rango).
START = pd.Timestamp("2023-06-01")
END = pd.Timestamp("2024-08-01")

COUNTRY = "ES"
TZ_DISPLAY = "Europe/Madrid"

SERIES_COLORS = plots.series_color_map(DEMO_SERIES_IDS)
print(SERIES_COLORS)


def save_figure(fig, name: str) -> Path:
    """Guarda una figura en reports/figures/ con un nombre estable y la muestra."""
    path = FIGURES_DIR / f"{name}.png"
    fig.write_image(path, width=1100, height=fig.layout.height or 500, scale=2)
    fig.show()
    return path

{'residential_north': '#2a78d6', 'commercial_mixed': '#eb6834', 'volatile_industrial': '#1baf7a'}


## Carga de datos: crudo → alineado

Se instancian las dos fuentes envueltas en `CachedSource`, apuntando a `data/raw/`: la primera llamada calcula y escribe en caché; cualquier reejecución posterior de esta notebook lee de disco sin recalcular nada, que es exactamente lo que exige "sin red, usando la caché".

El resultado de `fetch()` es la trama **cruda**: en hora local de Madrid, con las marcas de tiempo duplicadas/ausentes propias del cambio de hora más las imperfecciones inyectadas a propósito (huecos, duplicados, ceros, atípicos). El resto de esta celda aplica el pipeline real de `chronolab.data.align` — `to_utc_naive` → descartar `NaT` → `deduplicate` → `reindex_to_full_grid` — para llegar a la trama **alineada** que usa el resto de la notebook. Es el mismo camino por el que pasaría UCI o REE.

In [2]:
demand_source = CachedSource(
    SyntheticElectricitySource(seed=0), cache_dir=REPO_ROOT / "data" / "raw"
)
weather_source = CachedSource(SyntheticWeatherSource(), cache_dir=REPO_ROOT / "data" / "raw")

raw_demand = demand_source.fetch(start=START, end=END)
raw_weather = weather_source.fetch(start=START, end=END)

print(f"demanda cruda: {len(raw_demand):,} filas, {raw_demand['unique_id'].nunique()} series")
print(f"clima crudo:   {len(raw_weather):,} filas")
raw_demand.head()

demanda cruda: 30,675 filas, 3 series
clima crudo:   10,248 filas


,unique_id,ds,y
0,residential_north,2023-06-01 02:00:00,128.804
1,residential_north,2023-06-01 03:00:00,129.257
2,residential_north,2023-06-01 04:00:00,127.979
3,residential_north,2023-06-01 05:00:00,128.331
4,residential_north,2023-06-01 06:00:00,132.939


In [3]:
# Demanda: hora local de Madrid -> UTC ingenuo -> deduplicar -> rejilla completa.
demand = raw_demand.copy()
demand["ds"] = to_utc_naive(demand["ds"], source_tz=TZ_DISPLAY, group=demand["unique_id"])
demand = demand.dropna(subset=["ds"])  # salto de primavera: hora local inexistente
demand = deduplicate(demand, policy="mean")
demand = reindex_to_full_grid(demand, freq="h")

# Clima: ya viene en UTC (declarado como tal en su SourceSpec), solo hace
# falta rejilla completa.
weather = reindex_to_full_grid(raw_weather, freq="h")

print(f"demanda alineada: {len(demand):,} filas, rejilla completa por serie")
print(f"clima alineado:   {len(weather):,} filas")
demand.head()

demanda alineada: 30,738 filas, rejilla completa por serie
clima alineado:   10,248 filas


,unique_id,ds,y
0,residential_north,2023-06-01 00:00:00,128.804
1,residential_north,2023-06-01 01:00:00,129.257
2,residential_north,2023-06-01 02:00:00,127.979
3,residential_north,2023-06-01 03:00:00,128.331
4,residential_north,2023-06-01 04:00:00,132.939


Se añaden las features de calendario (`chronolab.data.calendar.calendar_features`, leídas en hora local de Madrid, festivos de España) y se cruza la temperatura horaria por `ds`, para tener una única trama de análisis con todo lo que hace falta en las secciones siguientes.

In [4]:
# Una fila de calendario por timestamp unico (no por (serie, timestamp)): la
# rejilla es la misma para las tres series.
calendar = calendar_features(
    demand.loc[demand["unique_id"] == DEMO_SERIES_IDS[0], "ds"].reset_index(drop=True),
    tz_display=TZ_DISPLAY,
    country=COUNTRY,
)

analysis = demand.merge(calendar, on="ds", how="left").merge(
    weather[["ds", "temp_c"]], on="ds", how="left"
)
analysis = analysis.sort_values(["unique_id", "ds"]).reset_index(drop=True)

# Series indexadas por tiempo, una por unique_id: la forma que esperan los
# compute_* de chronolab.viz.plots.
series_by_id = {
    series_id: analysis.loc[analysis["unique_id"] == series_id].set_index("ds")["y"]
    for series_id in DEMO_SERIES_IDS
}

analysis.head()

,unique_id,ds,y,hour,dayofweek,month,is_weekend,hour_sin,hour_cos,dow_sin,dow_cos,month_sin,month_cos,is_holiday,temp_c
0,commercial_mixed,2023-06-01 00:00:00,202.668,2,3,6,False,0.500,8.660e-01,0.434,-0.901,0.5,-0.866,False,19.997
1,commercial_mixed,2023-06-01 01:00:00,209.425,3,3,6,False,0.707,7.071e-01,0.434,-0.901,0.5,-0.866,False,20.415
2,commercial_mixed,2023-06-01 02:00:00,204.439,4,3,6,False,0.866,5.000e-01,0.434,-0.901,0.5,-0.866,False,19.978
3,commercial_mixed,2023-06-01 03:00:00,217.027,5,3,6,False,0.966,2.588e-01,0.434,-0.901,0.5,-0.866,False,18.468
4,commercial_mixed,2023-06-01 04:00:00,219.270,6,3,6,False,1.000,6.123e-17,0.434,-0.901,0.5,-0.866,False,21.833


## 1. Perfil de calidad de datos

`chronolab.data.quality.coverage_report` compara la trama cruda con la alineada y produce, por serie: cobertura temporal, huecos (`NaN` tras `reindex_to_full_grid`), duplicados (pares `(unique_id, ds)` repetidos en la cruda), ceros y atípicos (z-score robusto por mediana/MAD, umbral 4.0 — un filtro de cordura, no un detector de anomalías).

In [5]:
quality_report = coverage_report(raw_demand, demand)
quality_report

,unique_id,first_ds,last_ds,n_raw,n_duplicated_pairs,n_expected_grid,n_aligned,n_gaps,coverage,n_zeros,n_outliers
0,commercial_mixed,2023-06-01 02:00:00,2024-07-31 23:00:00,10225,21,10246,10246,41,0.996,72,80
1,residential_north,2023-06-01 02:00:00,2024-07-31 23:00:00,10225,21,10246,10246,41,0.996,0,8
2,volatile_industrial,2023-06-01 02:00:00,2024-07-31 23:00:00,10225,21,10246,10246,41,0.996,0,453


In [6]:
fig = plots.plot_quality_overview(quality_report)
save_figure(fig, "01_quality_overview")

WindowsPath('C:/Users/mpaez/Desktop/MATEO/.PROJECTS/.ProjectFiles/Chronolab Project/chronolab/reports/figures/01_quality_overview.png')

Serie a serie, con los huecos como cortes en la línea (nunca rellenados) y los atípicos marcados en rojo. `volatile_industrial` es la que más atípicos marca por z-score — conviene mirarla con cuidado antes de asumir que son errores de datos: los eventos de lote que la caracterizan son estructura real de la serie, no ruido, y un detector genérico no distingue una cosa de la otra sin más contexto.

In [7]:
outliers = detect_outliers(demand, z_threshold=4.0)
outlier_ds_by_series = outliers.groupby("unique_id")["ds"].apply(set).to_dict()


def despike(series_id: str, series: pd.Series) -> pd.Series:
    """Sustituye por NaN los puntos marcados como atipicos y vuelve a interpolar.

    Se reutiliza en las secciones 3 y 6: ambas calculan estadisticos basados
    en varianza o en autocorrelacion, que unos pocos picos extremos pueden
    distorsionar de forma desproporcionada.
    """
    flagged = outlier_ds_by_series.get(series_id, set())
    return series.where(~series.index.isin(flagged)).interpolate(limit_direction="both")


for series_id in DEMO_SERIES_IDS:
    series_frame = demand.loc[demand["unique_id"] == series_id]
    fig = plots.plot_series_with_flags(
        series_frame, outliers, unique_id=series_id, color=SERIES_COLORS[series_id]
    )
    save_figure(fig, f"01_series_flags_{series_id}")

### Continuidad tras el cambio de hora

La ventana cubre dos transiciones de DST reales: el vuelco de otoño del 29 de octubre de 2023 y el salto de primavera del 31 de marzo de 2024. `dst_transition_report` compara filas por día civil en la trama cruda (23 en el salto, 25 en el vuelco, si no hay imperfecciones que lo alteren) contra huecos/duplicados en la trama ya alineada — que deben ser cero en ambos casos si el pipeline funciona.

In [8]:
DST_TRANSITIONS = [pd.Timestamp("2023-10-29"), pd.Timestamp("2024-03-31")]

dst_report = dst_transition_report(raw_demand, demand, transitions=DST_TRANSITIONS)
dst_report

,transition,n_rows_local_day,n_duplicated_ds_aligned,n_gap_rows_aligned
0,2023-10-29,75,0,0
1,2024-03-31,70,0,0


In [9]:
for transition in DST_TRANSITIONS:
    fig = plots.plot_dst_continuity(
        demand,
        unique_id="residential_north",
        transition=transition,
        color=SERIES_COLORS["residential_north"],
    )
    save_figure(fig, f"01_dst_continuity_{transition.date()}")

## 2. Descomposición MSTL (24, 168)

`chronolab.viz.plots.compute_mstl` envuelve `statsmodels.tsa.seasonal.MSTL` con periodos `(24, 168)` — estacionalidad diaria y semanal a la vez, que es exactamente el caso de demanda horaria. Los huecos se interpolan antes de descomponer (MSTL no admite `NaN`); la descomposición es aditiva, así que `tendencia + estacional_24 + estacional_168 + residuo` reconstruye la serie observada.

Se descompone primero `residential_north` (la más estacional, la más fácil de leer) y después las otras dos para contraste.

In [10]:
MSTL_PERIODS = (24, 168)

mstl_components = {}
for series_id in DEMO_SERIES_IDS:
    components = plots.compute_mstl(series_by_id[series_id], periods=MSTL_PERIODS)
    mstl_components[series_id] = components
    fig = plots.plot_mstl(components, periods=MSTL_PERIODS)
    save_figure(fig, f"02_mstl_{series_id}")

## 3. ACF, PACF y periodograma

ACF y PACF hasta el retardo 200 horas (algo más de 8 días, suficiente para ver el pico semanal en 168) sobre `residential_north`. El periodograma reexpresa la densidad espectral en periodo (horas) en vez de frecuencia, con líneas verticales en 24 y 168 para confirmar visualmente que esos son los periodos dominantes — y no otro artefacto de la ventana de datos.

**Se usa la serie sin los puntos marcados como atípicos** (`despike`, definida en la Sección 1): la autocorrelación resultó ser aún más sensible a los picos inyectados que las razones de varianza de la Sección 6 — con los atípicos presentes, el ACF en el retardo 24 cae de 0.84 a 0.08, una estructura clarísima que un puñado de picos (8 de 10.246 puntos) esconde casi por completo. Vale la pena tenerlo presente: en datos reales, un par de lecturas erróneas sin limpiar pueden hacer parecer "sin estructura" una serie que en realidad es muy predecible.

Se repite el periodograma para `volatile_industrial` como contraste: si el diseño de la serie es el que se pretende, su espectro debería verse mucho más plano.

In [11]:
MAX_LAG = 200
primary_series = despike("residential_north", series_by_id["residential_north"])

acf_pacf_table = plots.compute_acf_pacf(primary_series, max_lag=MAX_LAG)
fig = plots.plot_acf_pacf(acf_pacf_table, n_obs=len(primary_series))
save_figure(fig, "03_acf_pacf_residential_north")

WindowsPath('C:/Users/mpaez/Desktop/MATEO/.PROJECTS/.ProjectFiles/Chronolab Project/chronolab/reports/figures/03_acf_pacf_residential_north.png')

In [12]:
periodogram_table = plots.compute_periodogram(primary_series)
fig = plots.plot_periodogram(periodogram_table, highlight_periods=(24.0, 168.0))
save_figure(fig, "03_periodogram_residential_north")

dominant_period = periodogram_table.loc[periodogram_table["power"].idxmax(), "period"]
print(f"periodo dominante (residential_north): {dominant_period:.1f} horas")

periodo dominante (residential_north): 24.0 horas


El periodo dominante es 24h, pero por muy poco: el segundo pico, en 12h, tiene casi la misma potencia (14.1 frente a 14.9). Esto no es ruido — es la forma exacta de la curva diaria de `residential_north`, con dos jorobas (mañana y noche), que concentra energía real en el segundo armónico. Antes de descartar los atípicos, el orden se invertía (12h por delante de 24h): otra forma en la que unos pocos picos extremos pueden desplazar cuál periodo parece "el dominante", no solo cuánta autocorrelación hay.

In [13]:
# Contraste: el periodograma de la serie debilmente estacional debe verse
# mucho mas plano (potencia repartida entre muchas frecuencias). Tambien
# despicada, por la misma razon que la serie principal.
industrial_series = despike("volatile_industrial", series_by_id["volatile_industrial"])
periodogram_industrial = plots.compute_periodogram(industrial_series)
fig = plots.plot_periodogram(periodogram_industrial, highlight_periods=(24.0, 168.0))
save_figure(fig, "03_periodogram_volatile_industrial")

WindowsPath('C:/Users/mpaez/Desktop/MATEO/.PROJECTS/.ProjectFiles/Chronolab Project/chronolab/reports/figures/03_periodogram_volatile_industrial.png')

## 4. Perfiles agregados

Heatmap hora del día × día de la semana para `residential_north` (perfil doméstico: mañana + noche) y `commercial_mixed` (perfil de horario comercial, casi apagado el fin de semana), para que el contraste de forma sea visible. Después, el efecto de festivo (tratado como fin de semana en el generador sintético) y el perfil mensual de `residential_north`, en el que debería verse la respuesta térmica como dos jorobas — invierno y verano — más que como una única tendencia suave.

In [14]:
for series_id in ("residential_north", "commercial_mixed"):
    series_analysis = analysis.loc[analysis["unique_id"] == series_id]
    matrix = plots.compute_hour_dow_matrix(series_analysis)
    fig = plots.plot_hour_dow_heatmap(matrix)
    save_figure(fig, f"04_heatmap_hour_dow_{series_id}")

In [15]:
for series_id in ("residential_north", "commercial_mixed"):
    series_analysis = analysis.loc[analysis["unique_id"] == series_id]
    fig = plots.plot_holiday_effect(series_analysis)
    save_figure(fig, f"04_holiday_effect_{series_id}")

In [16]:
residential_analysis = analysis.loc[analysis["unique_id"] == "residential_north"]
monthly_table = plots.compute_monthly_profile(residential_analysis)
fig = plots.plot_monthly_profile(monthly_table)
save_figure(fig, "04_monthly_profile_residential_north")

WindowsPath('C:/Users/mpaez/Desktop/MATEO/.PROJECTS/.ProjectFiles/Chronolab Project/chronolab/reports/figures/04_monthly_profile_residential_north.png')

## 5. Relación demanda-temperatura

Se espera una forma en **U**: la demanda sube con el frío (calefacción) y con el calor (refrigeración), con un mínimo en la zona de confort térmico. El ajuste es LOWESS (no paramétrico, `chronolab.viz.plots.compute_lowess_fit`) para no imponer la forma de antemano — si la relación no fuese en U, el ajuste no la fabricaría.

Después se calculan grados-día de calefacción (HDD, base 18°C) y refrigeración (CDD, base 22°C) a partir de la temperatura diaria media, y se correlacionan con la demanda diaria media: es la forma estándar de cuantificar la respuesta térmica sin depender de la forma exacta de la curva.

In [17]:
for series_id in ("residential_north", "commercial_mixed"):
    series_analysis = analysis.loc[analysis["unique_id"] == series_id]
    fit = plots.compute_lowess_fit(series_analysis["temp_c"], series_analysis["y"], frac=0.15)
    fig = plots.plot_temperature_scatter(
        series_analysis,
        fit,
        color=SERIES_COLORS[series_id],
        title=f"{series_id}: demanda frente a temperatura",
    )
    save_figure(fig, f"05_temp_scatter_{series_id}")

In [18]:
degree_days = plots.compute_degree_days(weather, base_heating=18.0, base_cooling=22.0)

for series_id in ("residential_north", "commercial_mixed"):
    series_frame = demand.loc[demand["unique_id"] == series_id]
    daily_demand = plots.compute_resampled_mean(series_frame, freq="D", value_column="y")
    fig = plots.plot_degree_days_correlation(degree_days, daily_demand)
    save_figure(fig, f"05_degree_days_correlation_{series_id}")

## 6. Estadísticos de dificultad de la serie

Tres estadísticos, uno por serie:

- **Fuerza de tendencia** y **fuerza estacional** (Hyndman & Wang 2015, generalizada a MSTL): comparan la varianza del residuo con la varianza de "residuo + componente". Cerca de 1 significa que el componente domina la serie; cerca de 0, que no aporta nada. Una fuerza estacional alta predice que un baseline estacional (`SeasonalNaive`, `MSTL`) va a ser difícil de batir — y que vale la pena tenerlo como referencia antes de complicar el modelo.
- **Entropía espectral**: entropía de Shannon de la densidad espectral normalizada, acotada a `[0, 1]`. Cerca de 0 es un espectro concentrado en pocas frecuencias (muy predecible); cerca de 1 es un espectro plano, indistinguible de ruido blanco. Es la métrica que mejor anticipa qué serie va a costar más trabajo en la Fase 4, independientemente de si esa dificultad viene de la estacionalidad o de otra estructura.

Las tres series sintéticas se diseñaron con una dificultad escalonada a propósito: esta tabla es la que debería confirmarlo — o desmentirlo, si el diseño no salió como se pretendía.

**Aviso metodológico que se descubrió al ejecutar esto, no al diseñarlo:** estos estadísticos son razones de varianzas, y la varianza es cuadrática en la desviación — unos pocos atípicos extremos bastan para dominarla. Se calcula la tabla dos veces, con y sin los puntos que `detect_outliers` marcó, para que el efecto quede documentado en vez de escondido detrás de un número que parece más bajo de lo que la serie realmente es.

In [19]:
interpolated_series = {
    series_id: series.interpolate(limit_direction="both")
    for series_id, series in series_by_id.items()
}
despiked_series = {
    series_id: despike(series_id, series) for series_id, series in series_by_id.items()
}

difficulty_table = plots.compute_difficulty_table(interpolated_series, periods=MSTL_PERIODS)
difficulty_table_despiked = plots.compute_difficulty_table(despiked_series, periods=MSTL_PERIODS)

print("con atipicos (tal cual llega de coverage_report / detect_outliers):")
display(difficulty_table)
print("\nsin los puntos marcados como atipicos:")
display(difficulty_table_despiked)

con atipicos (tal cual llega de coverage_report / detect_outliers):


,unique_id,trend_strength,seasonal_strength_24,seasonal_strength_168,spectral_entropy
0,residential_north,0.040,0.273,0.153,0.932
1,commercial_mixed,0.278,0.387,0.262,0.776
2,volatile_industrial,0.066,0.188,0.147,0.927



sin los puntos marcados como atipicos:


,unique_id,trend_strength,seasonal_strength_24,seasonal_strength_168,spectral_entropy
0,residential_north,0.674,0.868,0.248,0.427
1,commercial_mixed,0.771,0.895,0.777,0.324
2,volatile_industrial,0.096,0.204,0.158,0.889


In [20]:
# Se guarda la version despicada como la tabla de referencia: es la que
# describe la dificultad estructural de cada serie sin el artefacto de unos
# pocos picos inyectados. La version "con atipicos" queda impresa arriba,
# no oculta, para que quien lea la notebook vea ambas.
fig = plots.plot_difficulty_table(difficulty_table_despiked)
save_figure(fig, "06_difficulty_table")

WindowsPath('C:/Users/mpaez/Desktop/MATEO/.PROJECTS/.ProjectFiles/Chronolab Project/chronolab/reports/figures/06_difficulty_table.png')

## Cierre

Los hallazgos de esta ejecución están recogidos en [`docs/EDA_FINDINGS.md`](../docs/EDA_FINDINGS.md), con el aviso de que son sobre datos sintéticos repetido en su cabecera. Cuando haya red disponible, el primer paso de la Fase 4 (motor de backtesting) debería ser volver a ejecutar esta misma notebook sustituyendo las fuentes sintéticas por `UCIElectricitySource` y `OpenMeteoSource`/`REEDemandSource` reales, y comparar si la estructura encontrada aquí (estacionalidad doble, forma en U, dificultad escalonada) se sostiene con datos de verdad.

Todas las figuras quedaron guardadas en `reports/figures/` con nombres estables (`<sección>_<figura>[_<serie>].png`), listas para citarse desde la documentación sin tener que reabrir esta notebook.